#### Notebook to fetch data from Storage bucket, process and upload to dashboard. 

In [1]:
from pathlib import Path
from spending_tracker.data_processing.statements import parse_and_clean_batch
from spending_tracker.database import transactions as transactions_db
from spending_tracker.database import supabase_client
from spending_tracker.data_ingestion import csv_parsing

In [2]:
data_path = Path().resolve().parent / "data"
month = "September-26"
bucket_name = "spending-tracker"

In [3]:
bucket = supabase_client.load_bucket(bucket_name)

#### Clean Transactions and generate Metadata

In [4]:
data_fetched = transactions_db.fetch_transactions_from_bucket(month=month,
                                               bucket=bucket,
                                               data_home=data_path)

for key in data_fetched:
    print(key)

data/transactions/raw/Joint/Nationwide/03076275/September-26/statement.csv
data/transactions/raw/Radhika/Nationwide/34490057/September-26/statement.csv
data/transactions/raw/Taj/Amex/52006/September-26/statement.csv
data/transactions/raw/Taj/Chase/33053200/September-26/statement.csv
data/transactions/raw/Taj/Chase/69448147/September-26/statement.csv


In [4]:
parse_and_clean_batch(data_home=data_path,
                      specific_month=month,
                      specific_bank="Chase")

Processing Taj/Chase/33053200/September-26
Processing Taj/Chase/69448147/September-26


#### Upload to Database

In [5]:
db_client = supabase_client.load_supabase_client()

In [6]:
processed_transactions_data_home = data_path / "transactions" / "processed"
transaction_metadata_home = data_path / "transactions_metadata"

In [7]:
transactions_df = csv_parsing.load_csv_in_batch(processed_transactions_data_home,
                                                specific_month=month,
                                                dtype_kwargs={
                                                    "account_id": str,
                                                })

transactions_metadata_df = csv_parsing.load_csv_in_batch(transaction_metadata_home,
                                                         specific_month=month,
                                                         dtype_kwargs={
                                                             "account_id": str,
                                                         })

In [8]:
transactions_df.head(10)

,event_id,date,month,account_id,user,bank,merchant,other_details,type,amount
0,bd111b5020ee5a1518b14556646d5602093349e4388f1f...,2026-09-01,September-26,03076275,Joint,Nationwide,DOMINO S PIZZA MILTON KEYNES GB,APPLEPAY 4914,NaN,-6.48
1,052ebaee9177cb9a0f194c620602fc7f808fe6bd08f08a...,2026-09-02,September-26,03076275,Joint,Nationwide,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,APPLEPAY 4914,NaN,-0.79
2,f6a0625d165905b80279e59e6dea4dea5f09c04362d545...,2026-09-04,September-26,03076275,Joint,Nationwide,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,APPLEPAY 4914,NaN,-3.00
3,c079609fd772bcb05b9f67e1abad6af5c87e3fbb5792b4...,2026-09-04,September-26,03076275,Joint,Nationwide,ICELAND KINGSBURY GB,APPLEPAY 4914,NaN,-3.85
4,e5790b4c063f3ab2f143b4142ac15c5466373ae30291ff...,2026-09-04,September-26,03076275,Joint,Nationwide,TESCO STORES 2126 BOREHAMWOOD GB,APPLEPAY 9160,NaN,-9.20
5,cc3dff86837e9462f2571c0e5a4015bc6bf87e92d39176...,2026-09-05,September-26,03076275,Joint,Nationwide,SQ *LITTLE LION ENTERTAIN London GB,APPLEPAY 4914,NaN,-15.00
6,6a3b99555370a760fa85222e3aa6e72849401e9df8ef40...,2026-09-07,September-26,03076275,Joint,Nationwide,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,APPLEPAY 9160,NaN,-5.10
7,6e49e5d592b2e9c4e6651a6dd21b20ead0e8668abc3165...,2026-09-07,September-26,03076275,Joint,Nationwide,MORRISONS COLINDALE - 373 COLINDALE GB,APPLEPAY 9160,NaN,-20.36
8,a20a2c9b3b8d455290c4a7b9eb288c9acbb09d0146aa5a...,2026-09-12,September-26,03076275,Joint,Nationwide,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,APPLEPAY 4914,NaN,-5.96
9,3cf06dce5025451e49cf6974ee94393b89360eb868c40d...,2026-09-14,September-26,03076275,Joint,Nationwide,WAITROSE STORES HARROW WEALD GB,APPLEPAY 9160,NaN,-10.46


In [9]:
transactions_metadata_df.head(10)

,event_id,llm_merchant,llm_category,manual_merchant,manual_category,manual_amount,hidden
0,bd111b5020ee5a1518b14556646d5602093349e4388f1f...,DOMINO S PIZZA MILTON KEYNES GB,dining,NaN,NaN,NaN,False
1,052ebaee9177cb9a0f194c620602fc7f808fe6bd08f08a...,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,groceries,NaN,NaN,NaN,False
2,f6a0625d165905b80279e59e6dea4dea5f09c04362d545...,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,groceries,NaN,NaN,NaN,False
3,c079609fd772bcb05b9f67e1abad6af5c87e3fbb5792b4...,ICELAND KINGSBURY GB,groceries,NaN,NaN,NaN,False
4,e5790b4c063f3ab2f143b4142ac15c5466373ae30291ff...,TESCO STORES 2126 BOREHAMWOOD GB,groceries,NaN,NaN,NaN,False
5,cc3dff86837e9462f2571c0e5a4015bc6bf87e92d39176...,SQ *LITTLE LION ENTERTAIN London GB,dining,NaN,NaN,NaN,False
6,6a3b99555370a760fa85222e3aa6e72849401e9df8ef40...,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,groceries,NaN,NaN,NaN,False
7,6e49e5d592b2e9c4e6651a6dd21b20ead0e8668abc3165...,Morrisons,groceries,NaN,NaN,NaN,False
8,a20a2c9b3b8d455290c4a7b9eb288c9acbb09d0146aa5a...,MORRISONS QUEENSBURY - 30 QUEENSBURY GB,groceries,NaN,NaN,NaN,False
9,3cf06dce5025451e49cf6974ee94393b89360eb868c40d...,Waitrose,groceries,NaN,NaN,NaN,False


In [10]:
transactions_df.shape

(132, 10)

In [11]:
transactions_metadata_df.shape

(132, 7)

In [12]:
transactions_db.upload_transactions_to_db(transactions_df, 
                                       db_client, 
                                       table_name="transactions")
transactions_db.upload_transactions_to_db(transactions_metadata_df, 
                                       db_client, 
                                       table_name="transactions_metadata",
                                       exclude_columns=["manual_merchant", "manual_category", "manual_amount", "manual_date", "hidden"])

APIResponse(data=[{'event_id': 'bd111b5020ee5a1518b14556646d5602093349e4388f1f3bbb0b36700de14a47', 'llm_merchant': 'DOMINO S PIZZA MILTON KEYNES GB', 'llm_category': 'dining', 'manual_merchant': None, 'manual_category': None, 'manual_amount': None, 'hidden': False, 'updated_at': '2026-10-01T12:44:32.176718+00:00'}, {'event_id': '052ebaee9177cb9a0f194c620602fc7f808fe6bd08f08a377eb9ccf0f64e9c9d', 'llm_merchant': 'MORRISONS QUEENSBURY - 30 QUEENSBURY GB', 'llm_category': 'groceries', 'manual_merchant': None, 'manual_category': None, 'manual_amount': None, 'hidden': False, 'updated_at': '2026-10-01T12:44:32.176718+00:00'}, {'event_id': 'f6a0625d165905b80279e59e6dea4dea5f09c04362d5456ffc49a2f325bc21ca', 'llm_merchant': 'MORRISONS QUEENSBURY - 30 QUEENSBURY GB', 'llm_category': 'groceries', 'manual_merchant': None, 'manual_category': None, 'manual_amount': None, 'hidden': False, 'updated_at': '2026-10-01T12:44:32.176718+00:00'}, {'event_id': 'c079609fd772bcb05b9f67e1abad6af5c87e3fbb5792b4a32